# First Names in Canada (2021 Census) Exploration

**Objective:** Explore the most common first names in Canada using a beginner-friendly workflow that downloads a ZIP file, extracts the CSV, and builds simple tables and charts.

**Dataset description:** This dataset comes from the 2021 Census of Population. Each row represents one first name in Canada. The file includes the total count, percentage, and rank for that name, along with separate values for `Men+` and `Women+`.

**What is a ZIP file?**
A ZIP file is a compressed folder. We cannot read the CSV inside it until we download the ZIP file and extract its contents.

**Data source(s):**
- Dataset page: https://www12.statcan.gc.ca/census-recensement/2021/dp-pd/names-noms/index.cfm?Lang=E
- ZIP download: https://www12.statcan.gc.ca/census-recensement/2021/dp-pd/names-noms/getDLFile.cfm?Lang=E&FileType=CSV

**Date / Author:** 2026-03-15 / Codex


## Step 1: Import our libraries

This cell brings in the tools we need for the notebook.

- `pandas` (`pd`) helps us work with table data.
- A **DataFrame** is a table object in pandas.
- We often name the DataFrame `df`, which is short for "data frame."
- `plotly.express` (`px`) helps us make charts.
- `Path`, `urlretrieve`, and `ZipFile` help us download and extract the dataset.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

import pandas as pd
import plotly.express as px


## Step 2: Set the download link and file paths

This cell stores the web address for the dataset and decides where we want to save the downloaded ZIP file and the extracted CSV file.

We use `Path(...)` to build file paths in a way that works cleanly across systems.


In [ ]:
download_url = "https://www12.statcan.gc.ca/census-recensement/2021/dp-pd/names-noms/getDLFile.cfm?Lang=E&FileType=CSV"

dataset_dir = Path("datasets/statcan-first-names-2021")
zip_path = dataset_dir / "98-505-X2021007_eng_CSV.zip"
csv_path = dataset_dir / "98-505-X2021007_eng_csv_data.csv"
meta_path = dataset_dir / "98-505-X2021007_eng_meta.txt"

dataset_dir.mkdir(parents=True, exist_ok=True)

zip_path, csv_path


## Step 3: Download the ZIP file

This cell downloads the compressed dataset from Statistics Canada into our local `datasets/` folder.

The `urlretrieve(...)` function saves the file directly to the path we give it.


In [ ]:
urlretrieve(download_url, zip_path)
zip_path.exists(), zip_path.stat().st_size


## Step 4: Look inside the ZIP file

This cell shows the names of the files stored inside the ZIP archive.

This helps us confirm the name of the CSV file before we extract it.


In [ ]:
with ZipFile(zip_path) as zipped_file:
    zipped_file.namelist()


## Step 5: Extract the files from the ZIP archive

This cell unpacks the ZIP file so the CSV and metadata text file become regular files in our folder.

After extraction, pandas can read the CSV from `csv_path`.


In [ ]:
with ZipFile(zip_path) as zipped_file:
    zipped_file.extractall(dataset_dir)

sorted(path.name for path in dataset_dir.iterdir())


## Step 6: Load the CSV and preview the rows

This cell reads the extracted CSV file into a DataFrame named `df` and shows the first few rows.

In `df.head()`, `head()` is a **method**. A method is an action we ask the DataFrame to do.


In [ ]:
df = pd.read_csv(csv_path)
df.head()


## Step 7: Display the column names

This cell shows the DataFrame's column labels so we know what information is available.

In `df.columns`, `columns` is a **property**. A property gives information about the DataFrame, instead of performing an action.


In [ ]:
display(df.columns)


## Step 8: Clean the table for analysis

This cell makes a cleaned copy of the table so we can do math and charts more easily.

- `copy()` creates a separate DataFrame so we do not accidentally change the original one.
- The file uses `...` when a value is not available. We replace that with `NaN`.
- `NaN` means "missing value" in pandas.
- `pd.to_numeric(..., errors="coerce")` converts text to numbers. If something cannot be converted, pandas turns it into `NaN` instead of crashing.


In [ ]:
cleaned_df = df.replace("...", pd.NA).copy()

numeric_columns = [
    "GENDER_TOTAL_COUNT",
    "GENDER_TOTAL_PERCENTAGE",
    "GENDER_TOTAL_RANK",
    "MEN+_COUNT",
    "MEN+_PERCENTAGE",
    "MEN+_RANK",
    "WOMEN+_COUNT",
    "WOMEN+_PERCENTAGE",
    "WOMEN+_RANK",
]

for column in numeric_columns:
    cleaned_df[column] = pd.to_numeric(cleaned_df[column], errors="coerce")

cleaned_df.head()


## Step 9: Find the top 20 names in Canada

This cell sorts the dataset by total count and keeps the 20 most common names.

The `nlargest(...)` method is a quick way to keep the biggest values in one column.


In [ ]:
top_20_names = cleaned_df.nlargest(20, "GENDER_TOTAL_COUNT")[[
    "FIRST_NAME",
    "GENDER_TOTAL_COUNT",
    "GENDER_TOTAL_RANK",
]]

top_20_names


## Step 10: Make a bar chart of the top 20 names

This cell turns the top 20 table into a bar chart so the differences are easier to see.

We sort the table so the chart reads from smaller bars at the bottom to larger bars at the top.


In [ ]:
chart_df = top_20_names.sort_values("GENDER_TOTAL_COUNT")

fig = px.bar(
    chart_df,
    x="GENDER_TOTAL_COUNT",
    y="FIRST_NAME",
    orientation="h",
    title="Top 20 first names in Canada by total count (2021 Census)",
    labels={"GENDER_TOTAL_COUNT": "Count", "FIRST_NAME": "First name"},
)
fig.show()


## Step 11: Find the top 10 `Women+` names

This cell keeps only rows where the `Women+` count is available, then finds the 10 largest values.

The `dropna(...)` method removes rows with missing values in the column we choose.


In [ ]:
top_women_names = cleaned_df.dropna(subset=["WOMEN+_COUNT"]).nlargest(10, "WOMEN+_COUNT")[[
    "FIRST_NAME",
    "WOMEN+_COUNT",
    "WOMEN+_RANK",
]]

top_women_names


## Step 12: Find the top 10 `Men+` names

This cell does the same type of filtering and ranking for the `Men+` column.

Because some names only appear in one gender column, we remove missing values first.


In [ ]:
top_men_names = cleaned_df.dropna(subset=["MEN+_COUNT"]).nlargest(10, "MEN+_COUNT")[[
    "FIRST_NAME",
    "MEN+_COUNT",
    "MEN+_RANK",
]]

top_men_names


## Step 13: Read the metadata note

This cell opens the metadata text file and shows the first part of it.

Metadata is information about the dataset, such as how it was collected, what was excluded, and what symbols mean.


In [ ]:
metadata_text = meta_path.read_text(encoding="utf-8", errors="replace")
print(metadata_text[:2000])


## Step 14: Reflection

This dataset is useful for ranking and comparison questions such as:
- Which names are most common in Canada overall?
- How do the `Men+` and `Women+` top 10 lists differ?
- Which names appear in only one of the gender columns?

A limitation is that this file is national only, so it does not let us compare provinces or territories.


---
### Open In Colab
[Open this notebook in Google Colab](https://colab.research.google.com/github/pbeens/ICD2O-OSC-2026-Project-Plan-Team-2/blob/main/notebooks/zip/first-names-canada-2021-exploration.ipynb)
